In [1]:
import pandas as pd
import sqlite3

# Load the cleaned data we saved yesterday
df = pd.read_csv('delivery_data_clean.csv')

# Create a SQLite database
conn = sqlite3.connect('delivery.db')

# Load the dataframe into SQL table
df.to_sql('deliveries', conn, if_exists='replace', index=False)

print("Database created!")
print("Table 'deliveries' loaded with", len(df), "rows")

Database created!
Table 'deliveries' loaded with 10000 rows


In [2]:
# SQL Query 1: What is the failure rate by zone?
query1 = """
SELECT 
    zone,
    COUNT(*) as total_trips,
    SUM(delivery_success) as successful_deliveries,
    ROUND(AVG(delivery_success) * 100, 2) as success_rate,
    ROUND((1 - AVG(delivery_success)) * 100, 2) as failure_rate
FROM deliveries
GROUP BY zone
ORDER BY failure_rate DESC;
"""

result1 = pd.read_sql_query(query1, conn)
print("=== Failure Rate by Zone ===")
print(result1)

=== Failure Rate by Zone ===
       zone  total_trips  successful_deliveries  success_rate  failure_rate
0     North         2021                   1701         84.17         15.83
1      East         1935                   1637         84.60         15.40
2  Downtown         2014                   1709         84.86         15.14
3     South         2047                   1742         85.10         14.90
4      West         1983                   1705         85.98         14.02


In [3]:
# SQL Query 2: Top 10 and bottom 10 drivers by success rate
query2 = """
SELECT 
    driver_id,
    COUNT(*) as total_trips,
    ROUND(AVG(delivery_success) * 100, 2) as success_rate,
    ROUND(AVG(delay_min), 2) as avg_delay,
    ROUND(AVG(speed_mph), 2) as avg_speed
FROM deliveries
GROUP BY driver_id
HAVING COUNT(*) >= 90
ORDER BY success_rate DESC
LIMIT 10;
"""

result2 = pd.read_sql_query(query2, conn)
print("=== Top 10 Drivers ===")
print(result2)

=== Top 10 Drivers ===
   driver_id  total_trips  success_rate  avg_delay  avg_speed
0         84           99         91.92       4.91      15.34
1         82          110         91.82       4.23      13.25
2         15          106         91.51       4.91      14.60
3         55           99         90.91       5.23      14.51
4         67           98         90.82       5.38      16.00
5         42          104         90.38       5.62      13.94
6         35          102         90.20       5.88      14.36
7         38          111         90.09       4.37      14.05
8         25          116         89.66       5.13      15.75
9         54          114         89.47       5.78      13.69


In [4]:
# Bottom 10 drivers
query3 = """
SELECT 
    driver_id,
    COUNT(*) as total_trips,
    ROUND(AVG(delivery_success) * 100, 2) as success_rate,
    ROUND(AVG(delay_min), 2) as avg_delay,
    ROUND(AVG(speed_mph), 2) as avg_speed
FROM deliveries
GROUP BY driver_id
HAVING COUNT(*) >= 90
ORDER BY success_rate ASC
LIMIT 10;
"""

result3 = pd.read_sql_query(query3, conn)
print("=== Bottom 10 Drivers ===")
print(result3)

=== Bottom 10 Drivers ===
   driver_id  total_trips  success_rate  avg_delay  avg_speed
0         85           94         76.60       6.02      15.13
1          4          103         77.67       4.68      14.17
2         77           92         78.26       4.93      15.05
3         88          103         78.64       5.48      13.38
4         24           95         78.95       4.96      14.95
5         86          106         79.25       4.59      15.58
6         87          121         79.34       5.25      16.27
7         27           97         79.38       6.56      13.80
8         22           98         79.59       4.66      13.60
9         46           93         80.65       5.42      14.78


In [5]:
# SQL Query 4: Which hours of the day have the most failures?
query4 = """
SELECT 
    hour,
    COUNT(*) as total_trips,
    ROUND(AVG(delivery_success) * 100, 2) as success_rate,
    ROUND(AVG(delay_min), 2) as avg_delay
FROM deliveries
GROUP BY hour
ORDER BY success_rate ASC
LIMIT 10;
"""

result4 = pd.read_sql_query(query4, conn)
print("=== Top 10 Worst Hours for Deliveries ===")
print(result4)

=== Top 10 Worst Hours for Deliveries ===
   hour  total_trips  success_rate  avg_delay
0    11          410         82.20       5.11
1     4          424         82.55       4.88
2    15          456         82.89       5.11
3     5          407         83.29       5.01
4     6          433         83.60       5.42
5    10          404         83.91       5.01
6     9          422         84.12       5.03
7    17          416         84.13       5.06
8    13          424         84.20       5.37
9    21          441         84.58       5.04


In [6]:
# SQL Query 5: Executive summary by zone and vehicle type
query5 = """
SELECT 
    zone,
    vehicle_type,
    COUNT(*) as total_trips,
    ROUND(AVG(delivery_success) * 100, 2) as success_rate,
    ROUND(AVG(delay_min), 2) as avg_delay,
    ROUND(AVG(speed_mph), 2) as avg_speed
FROM deliveries
GROUP BY zone, vehicle_type
ORDER BY zone, success_rate DESC;
"""

result5 = pd.read_sql_query(query5, conn)
print("=== Executive Summary: Zone x Vehicle Type ===")
print(result5)

=== Executive Summary: Zone x Vehicle Type ===
        zone vehicle_type  total_trips  success_rate  avg_delay  avg_speed
0   Downtown         Bike          605         86.28       4.90      14.45
1   Downtown          Car          610         84.43       5.03      15.77
2   Downtown          Van          799         84.11       5.35      14.76
3       East          Car          581         85.54       5.11      14.37
4       East         Bike          601         85.52       4.96      15.58
5       East          Van          753         83.13       5.20      14.88
6      North          Car          578         86.51       5.44      15.28
7      North          Van          843         83.51       5.05      14.69
8      North         Bike          600         82.83       5.03      14.90
9      South          Car          607         87.15       5.01      15.97
10     South         Bike          607         85.83       4.88      14.55
11     South          Van          833         83.07 

In [7]:
# Save all results for Tableau
result1.to_csv('zone_failure_rate.csv', index=False)
result2.to_csv('top_drivers.csv', index=False)
result3.to_csv('bottom_drivers.csv', index=False)
result4.to_csv('worst_hours.csv', index=False)
result5.to_csv('zone_vehicle_summary.csv', index=False)

# Close database connection
conn.close()

print("All results saved!")
print("\nFiles ready for Tableau:")
print("- zone_failure_rate.csv")
print("- top_drivers.csv")
print("- bottom_drivers.csv")
print("- worst_hours.csv")
print("- zone_vehicle_summary.csv")

All results saved!

Files ready for Tableau:
- zone_failure_rate.csv
- top_drivers.csv
- bottom_drivers.csv
- worst_hours.csv
- zone_vehicle_summary.csv
